# 金融统计补证计算
Codex已读央行8月/前三季度报告；每行单位、期间、口径及来源见配套JSON。只复核披露值之差，不能从累计差推官方单月实际数，不能把部门残差补成已知部门。

In [1]:
import json
from pathlib import Path
from decimal import Decimal
project = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
data = json.loads((project / 'data/financial_stats_comparisons_20261015.json').read_text(encoding='utf-8'))
rows = data['rows']
assert len({r['metric_id'] for r in rows}) == len(rows)
for r in rows:
    assert Decimal(r['current_value']) - Decimal(r['baseline_value']) == Decimal(r['reported_difference'])
    assert r['official_monthly_flow'] is None and r['surprise'] is None
by_id = {r['metric_id']: r for r in rows}
assert by_id['household_cumulative']['reported_difference'] == '0.00'
assert by_id['household_cumulative']['official_monthly_flow'] is None
for prefix, result in zip(('baseline', 'current'), data['checks']['sector_reconciliation']):
    shown = sum(Decimal(by_id[k][prefix+'_value']) for k in ('household_cumulative','enterprise_cumulative','nonbank_cumulative'))
    total = Decimal(by_id['loan_cumulative'][prefix+'_value'])
    assert total-shown == Decimal(result['unreconciled_difference'])
    assert result['partition_completeness_confirmed'] is False
print('16 metric comparisons reproduced; unreconciled sector coverage retained, not repaired.')


16 metric comparisons reproduced; unreconciled sector coverage retained, not repaired.
